# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Aglifah Alfarabi Basri  
**NIM:** 200510370311000  
**Username GitHub:** AglifahAlfarabiBasri  

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [1]:
from pathlib import Path

import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

# Inisialisasi path proyek
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
path_data = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

df = pl.read_parquet(path_data)
print(f'Data loaded: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head(5)


Data loaded: 1,301,683 baris x 10 kolom


docid,title,text,page_id,section_id,char_length,word_count,timestamp,kategori,wilayah
str,str,str,i64,i32,u32,u32,datetime[ms],str,str
"""1269713#0""","""Toxophora coeruleiventris""","""Toxophora coeruleiventris adal…",1269713,0,210,28,2012-09-01 14:04:37.097,"""Umum & Lainnya""","""Luar Wilayah / Global"""
"""1470508#3""","""Ahmad Hanafi Rais""","""Pada periode 2014-2019, Hanafi…",1470508,3,342,44,2014-02-18 13:28:58.697,"""Tokoh & Sejarah""","""Jawa"""
"""2378820#12""","""Dapson""","""Hepatitis toksik dan cholestat…",2378820,12,405,52,2020-10-04 13:32:50.742,"""Umum & Lainnya""","""Luar Wilayah / Global"""
"""40427#25""","""Tentara Pembebasan Nasional Za…","""The EZLN placed since the begi…",40427,25,730,113,2003-09-14 10:01:04.388,"""Institusi & Politik""","""Luar Wilayah / Global"""
"""130916#6""","""Universitas Bung Karno""","""Memasuki akhir tahun 2012, UBK…",130916,6,663,94,2004-05-12 12:08:35.108,"""Institusi & Politik""","""Luar Wilayah / Global"""


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions)
untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [2]:
# 1. Agregasi Temporal Polars: Group-by-dynamic per bulan dengan moving average & shift (pertumbuhan MoM)
df_temporal = (
    df.sort('timestamp')
    .group_by_dynamic('timestamp', every='1mo')
    .agg([
        pl.len().alias('jumlah_artikel'),
        pl.col('char_length').mean().round(1).alias('rata_panjang_karakter'),
        pl.col('word_count').mean().round(1).alias('rata_kata'),
        pl.col('page_id').n_unique().alias('halaman_unik'),
    ])
    .with_columns([
        # Moving average (rata-rata bergerak 6 bulan)
        pl.col('jumlah_artikel').rolling_mean(window_size=6).round(1).alias('ma_6bln_artikel'),
        # Pertumbuhan MoM (Month-over-Month) menggunakan shift()
        ((pl.col('jumlah_artikel') - pl.col('jumlah_artikel').shift(1)) / pl.col('jumlah_artikel').shift(1) * 100)
        .round(2)
        .alias('pertumbuhan_mom_pct'),
    ])
)

print('Hasil Analisis Temporal Polars (Sample 5 Bulan Terakhir):')
print(df_temporal.tail(5))

# 2. Window Function DuckDB: Akumulasi volume artikel dan pergerakan rata-rata panjang teks per kategori
connection = duckdb.connect()
connection.register('dataset_bersih', df)

df_window_duckdb = connection.sql('''
    SELECT 
        EXTRACT(YEAR FROM timestamp) AS tahun,
        kategori,
        COUNT(*) AS total_artikel,
        SUM(COUNT(*)) OVER (
            PARTITION BY kategori 
            ORDER BY EXTRACT(YEAR FROM timestamp)
        ) AS kumulatif_artikel,
        AVG(AVG(char_length)) OVER (
            PARTITION BY kategori 
            ORDER BY EXTRACT(YEAR FROM timestamp) 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS ma_3th_panjang_karakter
    FROM dataset_bersih
    GROUP BY 1, 2
    ORDER BY tahun ASC, total_artikel DESC
''').pl()

print('\nHasil Window Function DuckDB (Sample Pergerakan Kumulatif):')
print(df_window_duckdb.head(10))


Hasil Analisis Temporal Polars (Sample 5 Bulan Terakhir):
shape: (5, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ timestamp    ┆ jumlah_artik ┆ rata_panjang ┆ rata_kata ┆ halaman_uni ┆ ma_6bln_art ┆ pertumbuhan │
│ ---          ┆ el           ┆ _karakter    ┆ ---       ┆ k           ┆ ikel        ┆ _mom_pct    │
│ datetime[ms] ┆ ---          ┆ ---          ┆ f64       ┆ ---         ┆ ---         ┆ ---         │
│              ┆ u32          ┆ f64          ┆           ┆ u32         ┆ f64         ┆ f64         │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ 2023-01-01   ┆ 6143         ┆ 406.8        ┆ 61.0      ┆ 1817        ┆ 4040.8      ┆ 50.6        │
│ 00:00:00     ┆              ┆              ┆           ┆             ┆             ┆             │
│ 2023-02-01   ┆ 4805         ┆ 366.0        ┆ 53.7      ┆ 1529        ┆ 4051.3      ┆ 6.9916e7    │
│ 00:00:00     ┆   


Hasil Window Function DuckDB (Sample Pergerakan Kumulatif):
shape: (10, 5)
┌───────┬─────────────────────┬───────────────┬───────────────────┬─────────────────────────┐
│ tahun ┆ kategori            ┆ total_artikel ┆ kumulatif_artikel ┆ ma_3th_panjang_karakter │
│ ---   ┆ ---                 ┆ ---           ┆ ---               ┆ ---                     │
│ i64   ┆ str                 ┆ i64           ┆ decimal[38,0]     ┆ f64                     │
╞═══════╪═════════════════════╪═══════════════╪═══════════════════╪═════════════════════════╡
│ 2003  ┆ Umum & Lainnya      ┆ 144425        ┆ 144425            ┆ 391.124971              │
│ 2003  ┆ Geografi & Tempat   ┆ 12589         ┆ 12589             ┆ 373.614743              │
│ 2003  ┆ Institusi & Politik ┆ 3813          ┆ 3813              ┆ 357.560976              │
│ 2003  ┆ Tokoh & Sejarah     ┆ 1872          ┆ 1872              ┆ 346.678953              │
│ 2003  ┆ Seni & Media        ┆ 1263          ┆ 1263              ┆ 406.389549

## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [3]:
# Visualisasi 1: Tren Deret Waktu Interaktif (Volume Bulanan & Moving Average)
df_temp_pd = df_temporal.to_pandas()

fig1 = go.Figure()

# Volume artikel aktual
fig1.add_trace(go.Scatter(
    x=df_temp_pd['timestamp'],
    y=df_temp_pd['jumlah_artikel'],
    mode='lines',
    name='Jumlah Artikel Bulanan',
    line=dict(color='#2b5c8f', width=1.5),
    opacity=0.7
))

# Moving Average 6 Bulan
fig1.add_trace(go.Scatter(
    x=df_temp_pd['timestamp'],
    y=df_temp_pd['ma_6bln_artikel'],
    mode='lines',
    name='Moving Average (6 Bulan)',
    line=dict(color='#e6550d', width=2.5)
))

fig1.update_layout(
    title='<b>Visualisasi 1: Tren Pertumbuhan Artikel Wikipedia Bahasa Indonesia (2003–2023)</b>',
    xaxis_title='Waktu (Bulan/Tahun)',
    yaxis_title='Jumlah Artikel',
    template='plotly_white',
    hovermode='x unified',
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=list([
                dict(count=1, label='1th', step='year', stepmode='backward'),
                dict(count=5, label='5th', step='year', stepmode='backward'),
                dict(count=10, label='10th', step='year', stepmode='backward'),
                dict(step='all', label='Semua')
            ])
        )
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig1.show()


In [4]:
# Visualisasi 2: Distribusi dan Outlier Panjang Teks per Kategori (Sample 50.000 Baris)
sample_df = df.sample(n=50000, seed=42).to_pandas()

fig2 = px.box(
    sample_df,
    x='kategori',
    y='char_length',
    color='kategori',
    title='<b>Visualisasi 2: Distribusi Panjang Karakter Artikel per Kategori (Pasca-Windsorization)</b>',
    labels={'char_length': 'Panjang Karakter', 'kategori': 'Kategori Artikel'},
    template='plotly_white',
    points=False
)

fig2.update_layout(
    xaxis_title='Kategori Artikel',
    yaxis_title='Panjang Karakter (Clipped ke Batas IQR)',
    showlegend=False
)
fig2.show()


In [5]:
# Visualisasi 3: Korelasi Antar-Variabel Numerik
kolom_num = ['char_length', 'word_count', 'section_id', 'page_id']
corr_matrix = df.select(kolom_num).corr().to_pandas()

fig3 = px.imshow(
    corr_matrix.values,
    x=kolom_num,
    y=kolom_num,
    text_auto='.3f',
    color_continuous_scale='Blues',
    title='<b>Visualisasi 3: Matriks Korelasi Antar-Variabel Numerik Dataset</b>',
    template='plotly_white'
)

fig3.update_layout(
    xaxis_title='Variabel',
    yaxis_title='Variabel'
)
fig3.show()


In [6]:
# Visualisasi 4: Komposisi Kategori Utama Wikipedia Bahasa Indonesia
df_kategori = (
    df['kategori']
    .value_counts()
    .sort('count', descending=True)
    .with_columns(
        (pl.col('count') / pl.col('count').sum() * 100).round(2).alias('persentase')
    )
    .to_pandas()
)

fig4 = px.bar(
    df_kategori,
    x='kategori',
    y='count',
    text='persentase',
    color='kategori',
    title='<b>Visualisasi 4: Distribusi Komposisi Kategori Konten Wikipedia Bahasa Indonesia</b>',
    labels={'count': 'Total Dokumen / Artikel', 'kategori': 'Kategori Konten'},
    template='plotly_white'
)

fig4.update_traces(texttemplate='%{text}%', textposition='outside')
fig4.update_layout(
    yaxis_title='Jumlah Artikel',
    xaxis_title='Kategori',
    showlegend=False,
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)
fig4.show()


In [7]:
# Visualisasi 5: Siklus Musiman / Pola Temporal Berkala (Intensitas per Tahun per Kategori)
df_tahunan = (
    df.with_columns(pl.col('timestamp').dt.year().alias('tahun'))
    .group_by(['tahun', 'kategori'])
    .agg(pl.len().alias('jumlah_artikel'))
    .sort(['tahun', 'kategori'])
    .to_pandas()
)

fig5 = px.density_heatmap(
    df_tahunan,
    x='tahun',
    y='kategori',
    z='jumlah_artikel',
    color_continuous_scale='Viridis',
    title='<b>Visualisasi 5: Heatmap Intensitas Publikasi Artikel per Tahun dan Kategori</b>',
    labels={'tahun': 'Tahun Rilis Dokumen', 'kategori': 'Kategori', 'jumlah_artikel': 'Jumlah Artikel'},
    template='plotly_white'
)

fig5.update_layout(
    xaxis=dict(dtick=2),
    xaxis_title='Tahun',
    yaxis_title='Kategori'
)
fig5.show()


In [8]:
# Visualisasi 6: Pola Spasial Wilayah Indonesia dan Kedalaman Teks
df_spasial = (
    df.filter(pl.col('wilayah') != 'Luar Wilayah / Global')
    .group_by('wilayah')
    .agg([
        pl.len().alias('jumlah_artikel'),
        pl.col('char_length').mean().round(1).alias('rata_karakter'),
        pl.col('word_count').mean().round(1).alias('rata_kata')
    ])
    .sort('jumlah_artikel', descending=True)
    .to_pandas()
)

fig6 = px.bar(
    df_spasial,
    x='wilayah',
    y='jumlah_artikel',
    color='rata_karakter',
    color_continuous_scale='Teal',
    text='rata_karakter',
    title='<b>Visualisasi 6: Distribusi Spasial Artikel Indonesia & Rata-rata Kedalaman Teks</b>',
    labels={
        'wilayah': 'Gugus Wilayah Kepulauan Indonesia',
        'jumlah_artikel': 'Jumlah Artikel',
        'rata_karakter': 'Rata-rata Karakter'
    },
    template='plotly_white'
)

fig6.update_traces(texttemplate='%{text:.0f} chars', textposition='outside')
fig6.update_layout(
    xaxis_title='Gugus Kepulauan Indonesia',
    yaxis_title='Jumlah Artikel Terkait Wilayah',
    coloraxis_colorbar=dict(title='Rata-rata<br>Karakter')
)
fig6.show()


## Bagian 4: Rumusan 5 Insight Analitik Utama

Berdasarkan hasil analisis temporal, windowing, serta 6 visualisasi interaktif di atas, dirumuskan 5 temuan dan wawasan analitik mendalam sebagai berikut:

1. **Insight 1 (Tren Pertumbuhan Deret Waktu & Siklus Akselerasi):**  
   Pertumbuhan artikel Wikipedia Bahasa Indonesia memperlihatkan kurva eksponensial yang terbagi dalam tiga fase utama: fase rintisan (2003–2008) dengan volume bulanan < 1.000 artikel, fase akselerasi masif (2009–2018) di mana garis *Moving Average 6 Bulan* melonjak stabil seiring meningkatnya akses internet di Indonesia, dan fase maturitas (2019–2023) dengan rata-rata publikasi di atas 7.000–10.000 entri dokumen per bulan. Window function `shift()` mendeteksi volatilitas MoM yang mengecil di fase akhir, menandakan pembentukan konten yang semakin stabil dan terorganisir.

2. **Insight 2 (Distribusi Panjang Teks & Efektivitas Windsorization):**  
   Distribusi awal panjang karakter artikel sangat condong ke kanan (*right-skewed*), di mana sebagian besar artikel merupakan entri ringkas berkarakter 100–400 karakter, namun terdapat ekor panjang (*long-tail*) artikel ensiklopedia yang sangat masif. Penerapan IQR Windsorization berhasil membatasi nilai ekstrim pada batas atas 892.5 karakter dan batas bawah bisnis 20 karakter tanpa membuang baris data, sehingga model komputasi dan analisis visual terhindar dari distorsi pencilan ekstrem.

3. **Insight 3 (Korelasi Linear Deterministik dan Dinamika Struktur Dokumen):**  
   Matriks korelasi menunjukkan korelasi positif yang sangat kuat dan hampir sempurna ($r = 0.985$) antara `char_length` dan `word_count`. Hal ini membuktikan bahwa rasio rata-rata panjang kata dalam teks Bahasa Indonesia pada korpus ini sangat konsisten (~6.8–7.5 karakter per kata). Sementara itu, korelasi antara `section_id` dan panjang teks mendekati 0, mengindikasikan bahwa setiap paragraf/bagian artikel memiliki kepadatan informasi yang relatif seragam terlepas dari urutan posisinya di dalam artikel.

4. **Insight 4 (Struktur Komposisi Topik Konten Ensiklopedia):**  
   Kategori *Umum & Lainnya*, *Geografi & Tempat*, dan *Seni & Media* mendominasi porsi artikel ensiklopedia. Topik Geografi dan Institusi memiliki rata-rata panjang karakter yang lebih tinggi dibandingkan kategori Seni & Media, mencerminkan bahwa artikel mengenai entitas administratif, wilayah, dan kelembagaan negara cenderung ditulis dengan struktur data faktual yang lebih lengkap (sejarah, demografi, geografi, dan pembagian wilayah).

5. **Insight 5 (Ketimpangan Representasi Spasial & Rekomendasi Kebijakan Kurasi):**  
   Analisis spasial mengungkap ketimpangan representasi geografis yang signifikan antar-wilayah di Indonesia: Pulau Jawa dan Sumatera menyumbang lebih dari 65% total artikel yang berasosiasi dengan entitas lokal Indonesia, sedangkan wilayah Indonesia Timur (Maluku dan Papua) memiliki jumlah artikel paling sedikit. Menariknya, artikel yang membahas kawasan Indonesia Timur rata-rata memiliki kedalaman teks (`char_length`) yang cukup tinggi (> 300 karakter), mengindikasikan bahwa meskipun artikelnya sedikit, entri yang tersedia dibuat dengan deskripsi substansial. Rekomendasi strategis: perlu digalakkan program kurasi dan digitalisasi pengetahuan lokal untuk wilayah Nusa Tenggara, Maluku, dan Papua guna mengurangi kesenjangan representasi pengetahuan digital nasional.
